# Module 1: LangChain Agents

## Building Intelligent Agents with LangChain, LangGraph, and Ollama

---

**Author:** Alwajeez Tech Training Program  
**Prerequisites:** Python basics, understanding of LLMs  
**Environment:** Local machine with Ollama running at `localhost:11434`  
**Model:** `llama3.2:3b`

---

### What You Will Learn

In this lab, you will learn how to build AI agents using **LangChain** and **LangGraph** -- two of the most widely used frameworks for building LLM-powered applications.

Our existing project already has a custom-built ReAct agent in `api/agentic_server.py`. That server includes:
- A **ReActAgent** class that implements the Thought-Action-Observation loop manually
- An **OllamaClient** that communicates with the Ollama API directly via `httpx`
- A **ToolExecutor** that dispatches tool calls (calculator, web search, file operations, etc.)

In this notebook, we will learn how to accomplish the **same functionality** using LangChain, which gives us:
- A standardized interface for LLMs (`ChatOllama`)
- A decorator-based tool system (`@tool`)
- A prebuilt ReAct agent (`create_react_agent` from LangGraph)
- Stateful conversation memory (`MemorySaver`)
- A graph-based agent framework (`StateGraph`)

### Notebook Outline

1. **Setup & Ollama Connection** -- Install packages and verify connectivity
2. **ChatOllama Basics** -- Learn LangChain message types and LLM interaction patterns
3. **Custom Tools** -- Build tools using the `@tool` decorator
4. **ReAct Agent** -- Use LangGraph's prebuilt ReAct agent
5. **Multi-Tool Chaining** -- Combine multiple tools in a single agent run
6. **Stateful Agent with Memory** -- Add conversation persistence with `MemorySaver`
7. **Custom LangGraph from Scratch** -- Build a manual state graph with nodes and edges
8. **FastAPI Integration** -- Connect the LangChain agent to our existing API server
9. **Comparison: Custom vs LangChain Agent** -- Side-by-side evaluation

Let's get started!

---

## Section 1: Setup & Ollama Connection

Before we can use LangChain with a local LLM, we need to:
1. Install the required Python packages
2. Verify that Ollama is running and accessible
3. Test that the `llama3.2:3b` model responds correctly

**Important:** Make sure Ollama is running on your machine before executing the cells below. You can start it with `ollama serve` in a terminal, or by launching the Ollama desktop application.

In [30]:
# Install required packages from requirements.txt
# This installs LangChain, LangGraph, and their Ollama integration
%pip install -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [31]:
# === Imports ===
# We import everything we will need throughout this notebook.

# Standard library
import os
import json
import time
from datetime import datetime
from typing import Optional

# HTTP client -- used for raw API calls and verification
import httpx

# LangChain core message types
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)

# LangChain tool decorator
from langchain_core.tools import tool

# LangChain Ollama integration
from langchain_ollama import ChatOllama

# LangGraph -- prebuilt ReAct agent
from langgraph.prebuilt import create_react_agent

# LangGraph -- for building custom graphs
from langgraph.graph import StateGraph, MessagesState, START, END

# LangGraph -- memory / checkpointing
from langgraph.checkpoint.memory import MemorySaver

# --- Configuration ---
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME ="gpt-oss:20b-cloud" #"llama3.2:3b"

print("All imports successful!")
print(f"Ollama URL : {OLLAMA_BASE_URL}")
print(f"Model      : {MODEL_NAME}")

All imports successful!
Ollama URL : http://localhost:11434
Model      : gpt-oss:20b-cloud


In [32]:
# === Step 1: Verify Ollama is running ===
# We use httpx to make a raw HTTP request to the Ollama API.
# This is the same approach used in our custom OllamaClient in agentic_server.py.

print("Checking Ollama connection...")
try:
    response = httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10.0)
    response.raise_for_status()
    models = response.json().get("models", [])
    model_names = [m["name"] for m in models]
    print(f"Ollama is running! Available models: {model_names}")

    if any(MODEL_NAME in name for name in model_names):
        print(f"Model '{MODEL_NAME}' is available.")
    else:
        print(f"WARNING: Model '{MODEL_NAME}' not found. Pull it with: ollama pull {MODEL_NAME}")
except httpx.ConnectError:
    print("ERROR: Cannot connect to Ollama. Make sure it is running at", OLLAMA_BASE_URL)
except Exception as e:
    print(f"ERROR: {e}")

Checking Ollama connection...


Ollama is running! Available models: ['llama3.2:3b', 'gpt-oss:120b-cloud', 'qwen3:8b', 'gpt-oss:120b', 'gpt-oss:20b-cloud']
Model 'gpt-oss:20b-cloud' is available.


In [33]:
# === Step 2: Test ChatOllama with a basic invoke ===
# ChatOllama is LangChain's wrapper around the Ollama chat API.
# It provides a standardized interface that works the same way
# whether your backend is Ollama, OpenAI, Anthropic, or others.

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7,
)

# Simple test -- invoke with a plain string
response = llm.invoke("Hello! Tell me one interesting fact about Saudi Arabia.")

print("Type of response:", type(response).__name__)
print("Response content:")
print(response.content)

Type of response: AIMessage
Response content:
One intriguing fact about Saudi Arabia is that **it is the only country in the world that has no official national holidays outside of the Islamic calendar**. All public holidays in Saudi Arabia are based on the lunar Islamic calendar, such as Eid al-Fitr, Eid al-Adha, and the Islamic New Year. This unique approach reflects the country's deep-rooted cultural and religious traditions.


---

## Section 2: ChatOllama Basics

LangChain uses a **message-based** interface for chat models. Every interaction is represented as a list of messages, where each message has a **role** and **content**.

### LangChain Message Types

| Message Type | Role | Purpose |
|---|---|---|
| `SystemMessage` | `system` | Sets the behavior, personality, and instructions for the AI |
| `HumanMessage` | `user` | Represents input from the user |
| `AIMessage` | `assistant` | Represents a response from the AI |
| `ToolMessage` | `tool` | Contains the result of a tool execution |

This is very similar to the `Message` model in our custom server (`agentic_server.py`), but LangChain provides **typed Python classes** instead of raw dictionaries. This gives us better IDE support, validation, and a consistent API across different LLM providers.

### Three Ways to Call the LLM

- **`invoke()`** -- Send a single request and get the full response
- **`stream()`** -- Get the response token-by-token (useful for real-time UIs)
- **`batch()`** -- Send multiple requests in parallel

In [34]:
# === Using LangChain Message Types ===
# Instead of passing plain strings, we can pass structured messages.
# This gives us fine-grained control over the conversation.

messages = [
    SystemMessage(content=(
        "You are a helpful assistant working at Alwajeez Tech in Jeddah. "
        "You answer concisely in 2-3 sentences."
    )),
    HumanMessage(content="What is an AI agent?"),
]

response = llm.invoke(messages)

print("--- SystemMessage + HumanMessage ---")
print(f"AI Response: {response.content}")
print()

# Now let's continue the conversation by adding the AI's reply and a follow-up
messages.append(AIMessage(content=response.content))
messages.append(HumanMessage(content="How is that different from a chatbot?"))

response2 = llm.invoke(messages)

print("--- Follow-up with conversation history ---")
print(f"AI Response: {response2.content}")

--- SystemMessage + HumanMessage ---
AI Response: An AI agent is a system that perceives its environment through sensors, processes that information using artificial intelligence techniques, and takes actions via actuators to achieve a goal or optimize a reward. It can learn and adapt over time, making decisions that mimic or surpass human-like reasoning.

--- Follow-up with conversation history ---
AI Response: A chatbot is a specific type of AI agent focused on conversational interaction—it interprets text or voice input and replies accordingly. An AI agent is broader: it may sense, reason, act, and learn in any environment, not just dialogue, and can perform tasks like navigation, control, or decision‑making beyond chat.


In [35]:
# === Streaming Example ===
# Streaming returns chunks of the response as they are generated.
# This is essential for building responsive user interfaces where
# you want to show text appearing in real-time.

print("Streaming response:")
print("-" * 40)

for chunk in llm.stream("Explain the ReAct pattern in AI agents in simple terms."):
    # Each chunk contains a small piece of the response.
    # chunk.content is typically one or a few tokens.
    print(chunk.content, end="", flush=True)

print()  # newline at the end
print("-" * 40)
print("Streaming complete.")

Streaming response:
----------------------------------------
### What is ReAct?

**ReAct** is a *pattern* (i.e., a recipe) for building AI agents that can *think* and *act* in a loop.  
The name comes from two words:

| **Reason** | **Act** |
|------------|---------|
| *Think*: the agent writes down a short explanation of what it’s doing or will do next. | *Act*: the agent performs an action (e.g., ask a question, call an API, write code, etc.). |

The pattern is especially handy for large‑language‑model (LLM) agents that need to solve problems that involve multiple steps, like answering a user query that requires looking up data, performing calculations, or interacting with external tools.

---

## 1. The ReAct Cycle in Plain English

1. **Start with a goal**  
   The agent is told what it should achieve (e.g., “Find the population of Tokyo”).

2. **Reason → Act → Reason → Act …**  
   The agent alternates between two phases:

   | Phase | What happens? | Why it matters |
   |-------|

In [36]:
# === Batch Processing ===
# The batch() method sends multiple prompts in parallel.
# This is useful when you have several independent queries
# and want to process them efficiently.

prompts = [
    "What is LangChain? Answer in one sentence.",
    "What is LangGraph? Answer in one sentence.",
    "What is Ollama? Answer in one sentence.",
]

print("Sending batch of 3 prompts...")
start = time.time()
responses = llm.batch(prompts)
elapsed = time.time() - start

for i, resp in enumerate(responses):
    print(f"\n[{i+1}] {prompts[i]}")
    print(f"    -> {resp.content}")

print(f"\nBatch completed in {elapsed:.2f}s")

Sending batch of 3 prompts...

[1] What is LangChain? Answer in one sentence.
    -> LangChain is an open‑source framework that simplifies building applications powered by large language models by offering reusable components for prompting, memory handling, and integrations with data sources and APIs.

[2] What is LangGraph? Answer in one sentence.
    -> LangGraph is an open‑source framework that lets developers design, compose, and orchestrate multimodal language‑model applications with graph‑structured workflows and stateful interactions.

[3] What is Ollama? Answer in one sentence.
    -> Ollama is a lightweight, open‑source framework that lets you download, run, and manage large language models locally on your own hardware—via a simple command‑line interface and web UI—so you can experiment and deploy private AI inference without relying on cloud services.

Batch completed in 5.29s


---

## Section 3: Custom Tools

In agentic AI, **tools** are functions that an LLM can decide to call when it needs to perform actions that go beyond text generation -- like doing math, reading files, searching the web, or getting the current time.

### How Our Custom Server Handles Tools

In `agentic_server.py`, tools are defined as:
1. A `ToolDefinition` (Pydantic model with name, description, and parameter schema)
2. A `ToolExecutor` class with methods like `_calculator()`, `_get_current_time()`, etc.

### How LangChain Handles Tools

LangChain simplifies this with the **`@tool` decorator**. You write a normal Python function with:
- A **docstring** (becomes the tool description)
- **Type-annotated parameters** (become the parameter schema)
- A **return value** (becomes the observation)

LangChain automatically generates the JSON schema that the LLM needs to understand how to call the tool. This is much less boilerplate than our manual approach!

Let's build four tools that mirror the ones in our custom server.

In [37]:
# === Tool 1: Calculator ===
# Mirrors the _calculator method in ToolExecutor from agentic_server.py

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result.
    Supports basic arithmetic: addition (+), subtraction (-),
    multiplication (*), division (/), and parentheses.
    Example: '25 * 47 + 130'
    """
    try:
        # Safety check: only allow mathematical characters
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return f"Error: Expression contains invalid characters. Only numbers and +-*/.() are allowed."
        result = eval(expression)
        return f"Result: {result}"
    except ZeroDivisionError:
        return "Error: Division by zero."
    except Exception as e:
        return f"Error evaluating expression: {e}"


# === Tool 2: File Reader ===
# Mirrors the _file_operations (read) in ToolExecutor

@tool
def file_reader(file_path: str) -> str:
    """Read the contents of a file and return it as text.
    Provide the full or relative path to the file.
    Returns the file contents (limited to first 5000 characters).
    """
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read(5000)
        if len(content) == 5000:
            content += "\n... [truncated at 5000 characters]"
        return content
    except FileNotFoundError:
        return f"Error: File not found: {file_path}"
    except PermissionError:
        return f"Error: Permission denied: {file_path}"
    except Exception as e:
        return f"Error reading file: {e}"


# === Tool 3: Web Search (Simulated) ===
# Mirrors the _web_search placeholder in ToolExecutor

@tool
def web_search(query: str) -> str:
    """Search the web for current information. Returns search results
    for the given query. Use this when you need real-time data, news,
    or facts that may not be in your training data.
    """
    # This is a simulated search. In production, you would integrate
    # with a real search API like SerpAPI, Tavily, or Bing.
    simulated_results = {
        "results": [
            {
                "title": f"Search result 1 for: {query}",
                "snippet": f"This is a simulated search result about '{query}'. "
                           f"In production, this would return real web results.",
                "url": "https://example.com/result1"
            },
            {
                "title": f"Search result 2 for: {query}",
                "snippet": f"Another simulated result related to '{query}'. "
                           f"Integrate a real search API for actual data.",
                "url": "https://example.com/result2"
            },
        ]
    }
    return json.dumps(simulated_results, indent=2)


# === Tool 4: Get Current Time ===
# Mirrors the _get_current_time method in ToolExecutor

@tool
def get_current_time(timezone: str = "Asia/Riyadh") -> str:
    """Get the current date and time for a given timezone.
    Default timezone is Asia/Riyadh (Saudi Arabia).
    Examples of timezones: 'Asia/Riyadh', 'UTC', 'US/Eastern', 'Europe/London'.
    """
    try:
        from zoneinfo import ZoneInfo
        tz = ZoneInfo(timezone)
        now = datetime.now(tz)
        return f"Current time in {timezone}: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except KeyError:
        return f"Error: Unknown timezone '{timezone}'. Use standard timezone names like 'Asia/Riyadh' or 'UTC'."
    except Exception as e:
        # Fallback to local time
        now = datetime.now()
        return f"Current local time: {now.strftime('%Y-%m-%d %H:%M:%S')} (timezone '{timezone}' unavailable: {e})"


# Collect all four tools into a list
basic_tools = [calculator, file_reader, web_search, get_current_time]

print("Tools defined:")
for t in basic_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Tools defined:
  - calculator: Evaluate a mathematical expression and return the result.
Supports basic arithme...
  - file_reader: Read the contents of a file and return it as text.
Provide the full or relative ...
  - web_search: Search the web for current information. Returns search results
for the given que...
  - get_current_time: Get the current date and time for a given timezone.
Default timezone is Asia/Riy...


In [38]:
# === Test Each Tool Directly ===
# Before giving tools to an agent, it's good practice to test them
# individually to make sure they work correctly.

print("=== Testing calculator ===")
print(calculator.invoke({"expression": "25 * 47 + 130"}))
print()

print("=== Testing file_reader ===")
print(file_reader.invoke({"file_path": "requirements.txt"}))
print()

print("=== Testing web_search ===")
print(web_search.invoke({"query": "latest AI news 2025"}))
print()

print("=== Testing get_current_time ===")
print(get_current_time.invoke({"timezone": "Asia/Riyadh"}))
print(get_current_time.invoke({"timezone": "UTC"}))

=== Testing calculator ===
Result: 1305

=== Testing file_reader ===
# Module 1: LangChain Agents
# Run locally - connects to Ollama at localhost:11434

langchain>=0.3.0
langchain-ollama>=0.3.0
langgraph>=0.2.0
langchain-core>=0.3.0
httpx>=0.25.0


=== Testing web_search ===
{
  "results": [
    {
      "title": "Search result 1 for: latest AI news 2025",
      "snippet": "This is a simulated search result about 'latest AI news 2025'. In production, this would return real web results.",
      "url": "https://example.com/result1"
    },
    {
      "title": "Search result 2 for: latest AI news 2025",
      "snippet": "Another simulated result related to 'latest AI news 2025'. Integrate a real search API for actual data.",
      "url": "https://example.com/result2"
    }
  ]
}

=== Testing get_current_time ===
Current time in Asia/Riyadh: 2026-02-24 04:23:12 +03
Current time in UTC: 2026-02-24 01:23:12 UTC


---

## Section 4: ReAct Agent

### What is the ReAct Pattern?

**ReAct** stands for **Re**asoning + **Act**ing. It is a prompting framework where the LLM follows a loop:

1. **Thought:** The LLM reasons about what to do next
2. **Action:** The LLM decides to call a tool with specific arguments
3. **Observation:** The tool returns a result, which the LLM observes
4. **Repeat** until the LLM has enough information to provide a final answer

### Custom vs LangChain Implementation

In our custom `agentic_server.py`, the ReAct loop is implemented manually:
- The `ReActAgent` class builds a system prompt that describes the Thought/Action/Observation format
- It parses the LLM output with regex to extract `ACTION: tool_name({...})`
- It executes the tool via `ToolExecutor` and feeds the result back
- It checks for `FINAL ANSWER:` to know when to stop

With **LangGraph**, all of this is handled automatically by `create_react_agent()`. The LLM uses **native tool calling** (structured JSON output) instead of text parsing, which is more reliable.

Let's see it in action!

In [39]:
# === Create a ReAct Agent with LangGraph ===
# This single function call replaces the entire ReActAgent class
# from agentic_server.py (around 100 lines of code!).

# The LLM needs to support tool calling. ChatOllama with llama3.2
# supports this natively.
llm_for_agent = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.0,  # Lower temperature for more deterministic tool use
)

# create_react_agent takes the LLM and a list of tools.
# It returns a compiled LangGraph that implements the full ReAct loop.
react_agent = create_react_agent(
    model=llm_for_agent,
    tools=basic_tools,
)

print("ReAct agent created successfully!")
print(f"Agent type: {type(react_agent).__name__}")

ReAct agent created successfully!
Agent type: CompiledStateGraph


C:\Users\renad\AppData\Local\Temp\ipykernel_8020\1998466026.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent = create_react_agent(


In [40]:
# === Test 1: Math Task ===
# The agent should use the calculator tool to solve this.

print("Task: What is 25 * 47 + 130?")
print("=" * 50)

result = react_agent.invoke(
    {"messages": [HumanMessage(content="What is 25 * 47 + 130?")]}
)

# The result contains the full message history
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n[{role}] Tool calls:")
        for tc in msg.tool_calls:
            print(f"  -> {tc['name']}({tc['args']})")
    elif hasattr(msg, "content") and msg.content:
        content_preview = msg.content[:300]
        print(f"\n[{role}] {content_preview}")

Task: What is 25 * 47 + 130?

[HumanMessage] What is 25 * 47 + 130?

[AIMessage] Tool calls:
  -> calculator({'expression': '25 * 47 + 130'})

[ToolMessage] Result: 1305

[AIMessage] 25 × 47 + 130 = 13 005.


In [41]:
# === Test 2: Time Task ===
# The agent should use the get_current_time tool.

print("Task: What time is it in Riyadh?")
print("=" * 50)

result = react_agent.invoke(
    {"messages": [HumanMessage(content="What time is it in Riyadh?")]}
)

for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n[{role}] Tool calls:")
        for tc in msg.tool_calls:
            print(f"  -> {tc['name']}({tc['args']})")
    elif hasattr(msg, "content") and msg.content:
        content_preview = msg.content[:300]
        print(f"\n[{role}] {content_preview}")

Task: What time is it in Riyadh?

[HumanMessage] What time is it in Riyadh?

[AIMessage] Tool calls:
  -> get_current_time({'timezone': 'Asia/Riyadh'})

[ToolMessage] Current time in Asia/Riyadh: 2026-02-24 04:23:20 +03

[AIMessage] The current time in Riyadh (Asia/Riyadh) is **2026‑02‑24 04:23:20 +03**.


In [42]:
# === Test 3: Web Search Task ===
# The agent should use the web_search tool.

print("Task: Search for latest AI news")
print("=" * 50)

result = react_agent.invoke(
    {"messages": [HumanMessage(content="Search for latest AI news")]}
)

for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n[{role}] Tool calls:")
        for tc in msg.tool_calls:
            print(f"  -> {tc['name']}({tc['args']})")
    elif hasattr(msg, "content") and msg.content:
        content_preview = msg.content[:500]
        print(f"\n[{role}] {content_preview}")

Task: Search for latest AI news

[HumanMessage] Search for latest AI news

[AIMessage] Tool calls:
  -> web_search({'query': 'latest AI news'})

[ToolMessage] {
  "results": [
    {
      "title": "Search result 1 for: latest AI news",
      "snippet": "This is a simulated search result about 'latest AI news'. In production, this would return real web results.",
      "url": "https://example.com/result1"
    },
    {
      "title": "Search result 2 for: latest AI news",
      "snippet": "Another simulated result related to 'latest AI news'. Integrate a real search API for actual data.",
      "url": "https://example.com/result2"
    }
  ]
}

[AIMessage] I’m sorry, but I don’t have real‑time browsing capabilities, so I can’t pull in fresh headlines from today’s web. Below is a quick snapshot of the major AI developments that were circulating around the time of my last knowledge update (end‑2023) and what you can expect going forward. If you need the absolute latest headlines, you can ru

---

## Section 5: Multi-Tool Chaining

A powerful capability of agents is **multi-step tool usage** -- where the agent uses multiple tools in sequence to accomplish a complex task. For example:

1. Use the **calculator** to compute a VAT amount
2. Use a **report formatter** to structure the result
3. Use a **file saver** to write the report to disk

The agent figures out the order on its own. It reasons about what it needs, calls the first tool, observes the result, then decides what to do next.

Let's add two new tools and test this multi-step workflow.

In [43]:
# === Additional Tools for Multi-Step Tasks ===

@tool
def save_to_file(filename: str, content: str) -> str:
    """Save text content to a file in the ./output/ directory.
    Provide just the filename (e.g., 'report.txt'), not the full path.
    The file will be saved to ./output/{filename}.
    """
    try:
        output_dir = "./output"
        os.makedirs(output_dir, exist_ok=True)
        filepath = os.path.join(output_dir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        return f"File saved successfully to {filepath}"
    except Exception as e:
        return f"Error saving file: {e}"


@tool
def format_report(title: str, content: str) -> str:
    """Format text content as a professional markdown report.
    Takes a title and content, returns a nicely formatted markdown report
    with title, timestamp, and formatted content.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    report = (
        f"# {title}\n"
        f"\n"
        f"**Generated:** {timestamp}  \n"
        f"**Author:** AI Agent (Alwajeez Tech)\n"
        f"\n"
        f"---\n"
        f"\n"
        f"{content}\n"
        f"\n"
        f"---\n"
        f"*Report generated automatically by AI Agent*\n"
    )
    return report


# Combine all 6 tools
all_tools = [calculator, file_reader, web_search, get_current_time, save_to_file, format_report]

print("All 6 tools:")
for t in all_tools:
    print(f"  - {t.name}")

All 6 tools:
  - calculator
  - file_reader
  - web_search
  - get_current_time
  - save_to_file
  - format_report


In [44]:
# === Create agent with all 6 tools ===

multi_tool_agent = create_react_agent(
    model=ChatOllama(
        model=MODEL_NAME,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
    ),
    tools=all_tools,
)

print("Multi-tool agent created with 6 tools.")

Multi-tool agent created with 6 tools.


C:\Users\renad\AppData\Local\Temp\ipykernel_8020\2279720916.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  multi_tool_agent = create_react_agent(


In [45]:
# === Multi-Tool Chaining Test ===
# This task requires the agent to:
# 1. Use calculator to compute 15% VAT on 2500 SAR
# 2. Use format_report to create a markdown report
# 3. Use save_to_file to write it to vat_report.txt

task = (
    "Calculate 15% VAT on 2500 SAR, then format the result "
    "as a professional report with the title 'VAT Calculation Report', "
    "and save it to vat_report.txt"
)

print(f"Task: {task}")
print("=" * 60)

result = multi_tool_agent.invoke(
    {"messages": [HumanMessage(content=task)]}
)

# Print the full chain of tool calls
print("\n--- Agent Execution Trace ---")
tool_call_count = 0
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            tool_call_count += 1
            print(f"\nStep {tool_call_count} - Tool Call: {tc['name']}")
            print(f"  Args: {json.dumps(tc['args'], indent=4)[:200]}")
    elif isinstance(msg, ToolMessage):
        content_short = msg.content[:200]
        print(f"  Result: {content_short}..." if len(msg.content) > 200 else f"  Result: {msg.content}")

# Print the final response
final_msg = result["messages"][-1]
print(f"\n--- Final Answer ---")
print(final_msg.content[:500])

# Verify the file was saved
if os.path.exists("./output/vat_report.txt"):
    print("\n--- Saved File Contents ---")
    with open("./output/vat_report.txt", "r") as f:
        print(f.read())

Task: Calculate 15% VAT on 2500 SAR, then format the result as a professional report with the title 'VAT Calculation Report', and save it to vat_report.txt

--- Agent Execution Trace ---

Step 1 - Tool Call: calculator
  Args: {
    "expression": "2500 * 15 / 100"
}
  Result: Result: 375.0

Step 2 - Tool Call: format_report
  Args: {
    "title": "VAT Calculation Report",
    "content": "**VAT Calculation**\n\n- **Base Amount**: 2,500 SAR\n- **VAT Rate**: 15%\n- **Calculated VAT**: 375 SAR\n\n**Summary**\n\nThe 15% Value Added T
  Result: # VAT Calculation Report

**Generated:** 2026-02-24 04:23:43  
**Author:** AI Agent (Alwajeez Tech)

---

**VAT Calculation**

- **Base Amount**: 2,500 SAR
- **VAT Rate**: 15%
- **Calculated VAT**: 37...

Step 3 - Tool Call: save_to_file
  Args: {
    "content": "# VAT Calculation Report\n\n**Generated:** 2026-02-24 04:23:43\n**Author:** AI Agent (Alwajeez Tech)\n\n---\n\n**VAT Calculation**\n\n- **Base Amount**: 2,500 SAR\n- **VAT Rate**: 15
  Result

---

## Section 6: LangGraph Stateful Agent with Memory

So far, our agent has been **stateless** -- each `invoke()` call starts a fresh conversation. The agent does not remember anything from previous interactions.

In many real-world applications, you need **conversation memory**. For example:
- A user says "My name is Ahmed" in turn 1
- In turn 2, they ask "What is my name?" -- the agent should remember

### How Memory Works in LangGraph

LangGraph provides a **checkpointer** system. A checkpointer saves the state of the conversation after each interaction. The key component is:

- **`MemorySaver`** -- An in-memory checkpointer (great for development and testing)
- **Thread IDs** -- Each conversation thread has a unique ID, so multiple users can have independent conversations

When you pass `config={"configurable": {"thread_id": "some-id"}}` to `invoke()`, LangGraph automatically saves and restores the conversation state for that thread.

This is a capability that our custom `ReActAgent` in `agentic_server.py` does **not** have -- it processes each request independently. Adding memory with LangGraph is just a few lines of code!

In [46]:
# === Create a Stateful Agent with Memory ===

# MemorySaver stores conversation state in memory (RAM).
# For production, you might use SqliteSaver or PostgresSaver.
memory = MemorySaver()

# Create the agent with the checkpointer
stateful_agent = create_react_agent(
    model=ChatOllama(
        model=MODEL_NAME,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
    ),
    tools=all_tools,
    checkpointer=memory,
)

print("Stateful agent created with MemorySaver checkpointer.")

Stateful agent created with MemorySaver checkpointer.


C:\Users\renad\AppData\Local\Temp\ipykernel_8020\3214840747.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  stateful_agent = create_react_agent(


In [47]:
# === Multi-Turn Conversation: Thread 1 ===
# We use thread_id to maintain conversation state across calls.

config_thread1 = {"configurable": {"thread_id": "thread-1"}}

# --- Turn 1: Introduction ---
print("=== Turn 1 ===")
result1 = stateful_agent.invoke(
    {"messages": [HumanMessage(content="My name is Ahmed and I work at Alwajeez Tech")]},
    config=config_thread1,
)
print(f"User: My name is Ahmed and I work at Alwajeez Tech")
print(f"Agent: {result1['messages'][-1].content}")

=== Turn 1 ===
User: My name is Ahmed and I work at Alwajeez Tech
Agent: Nice to meet you, Ahmed! How can I assist you today with Alwajeez Tech projects or any other topic you have in mind?


In [48]:
# --- Turn 2: Test memory recall ---
print("\n=== Turn 2 ===")
result2 = stateful_agent.invoke(
    {"messages": [HumanMessage(content="What is my name and where do I work?")]},
    config=config_thread1,
)
print(f"User: What is my name and where do I work?")
print(f"Agent: {result2['messages'][-1].content}")


=== Turn 2 ===
User: What is my name and where do I work?
Agent: Your name is **Ahmed**, and you work at **Alwajeez Tech**.


In [49]:
# --- Turn 3: Use a tool within the same conversation thread ---
print("\n=== Turn 3 ===")
result3 = stateful_agent.invoke(
    {"messages": [HumanMessage(content="Calculate 1000 * 0.15 for my VAT calculation")]},
    config=config_thread1,
)
print(f"User: Calculate 1000 * 0.15 for my VAT calculation")
# Show tool calls if any
for msg in result3["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [Tool Call] {tc['name']}({tc['args']})")
    elif isinstance(msg, ToolMessage):
        print(f"  [Tool Result] {msg.content}")
print(f"Agent: {result3['messages'][-1].content}")


=== Turn 3 ===
User: Calculate 1000 * 0.15 for my VAT calculation
  [Tool Call] calculator({'expression': '1000 * 0.15'})
  [Tool Result] Result: 150.0
Agent: The VAT calculation comes out to **150.0**.


In [50]:
# === Thread Isolation Test ===
# Using a DIFFERENT thread_id means a completely fresh conversation.
# The agent should NOT know Ahmed's name or workplace.

config_thread2 = {"configurable": {"thread_id": "thread-2"}}

print("=== Thread 2 (Separate Conversation) ===")
result_isolated = stateful_agent.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config_thread2,
)
print(f"User: What is my name?")
print(f"Agent: {result_isolated['messages'][-1].content}")
print()
print("Note: The agent should NOT know the name because this is a separate thread.")
print("This proves that thread isolation is working correctly.")

=== Thread 2 (Separate Conversation) ===
User: What is my name?
Agent: I’m not sure what your name is. Could you let me know?

Note: The agent should NOT know the name because this is a separate thread.
This proves that thread isolation is working correctly.


---

## Section 7: Custom LangGraph from Scratch

While `create_react_agent()` is convenient, sometimes you need **full control** over the agent's behavior. LangGraph lets you build a custom graph with:

- **Nodes** -- Functions that process the state (e.g., calling the LLM, executing tools)
- **Edges** -- Connections between nodes that define the flow
- **Conditional Edges** -- Branching logic (e.g., if the LLM wants to call a tool, go to the tool node; otherwise, end)
- **State** -- A shared data structure that flows through the graph

### Graph Structure

Our custom graph will look like this:

```
START -> call_model -> should_continue? --yes--> call_tools -> call_model -> ...
                                         |
                                         no--> END
```

This is exactly the same ReAct loop, but built manually so you can customize every step.

### Why Build It Manually?

- Add custom logging, metrics, or error handling between steps
- Implement custom routing logic (e.g., human-in-the-loop approval before tool execution)
- Add pre-processing or post-processing at any step
- Build non-standard agent patterns (not just ReAct)

In [51]:
# === Build a Custom LangGraph Agent from Scratch ===

# We'll use MessagesState, which is a built-in state schema
# that holds a list of messages (similar to how our custom agent
# maintains a messages list in agentic_server.py).

# Step 1: Create an LLM with tools bound to it.
# bind_tools() tells the LLM what tools are available.
llm_with_tools = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.0,
).bind_tools(basic_tools)


# Step 2: Define the "call_model" node.
# This node sends the current messages to the LLM and gets a response.
def call_model(state: MessagesState):
    """Node that calls the LLM with the current message history."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    # Return the response to be added to the state
    return {"messages": [response]}


# Step 3: Define the "call_tools" node.
# This node executes any tool calls the LLM made.
def call_tools(state: MessagesState):
    """Node that executes tool calls from the last AI message."""
    messages = state["messages"]
    last_message = messages[-1]

    # Build a lookup dictionary of our tools
    tool_map = {t.name: t for t in basic_tools}

    tool_results = []
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        print(f"  [Custom Graph] Executing tool: {tool_name}({tool_args})")

        # Execute the tool
        if tool_name in tool_map:
            result = tool_map[tool_name].invoke(tool_args)
        else:
            result = f"Error: Unknown tool '{tool_name}'"

        # Create a ToolMessage with the result
        tool_results.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
                name=tool_name,
            )
        )

    return {"messages": tool_results}


# Step 4: Define the conditional edge function.
# This decides whether to continue to tools or end.
def should_continue(state: MessagesState):
    """Decide whether to call tools or finish."""
    last_message = state["messages"][-1]
    # If the LLM made tool calls, we need to execute them
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "call_tools"
    # Otherwise, we're done
    return END


# Step 5: Build the graph
workflow = StateGraph(MessagesState)

# Add nodes
workflow.add_node("call_model", call_model)
workflow.add_node("call_tools", call_tools)

# Add edges
# START -> call_model (always start by calling the model)
workflow.add_edge(START, "call_model")

# call_model -> (conditional) either call_tools or END
workflow.add_conditional_edges("call_model", should_continue)

# call_tools -> call_model (after tools run, go back to the model)
workflow.add_edge("call_tools", "call_model")

# Step 6: Compile the graph
custom_graph = workflow.compile()

print("Custom LangGraph agent compiled successfully!")
print("Graph nodes:", list(workflow.nodes.keys()))

Custom LangGraph agent compiled successfully!
Graph nodes: ['call_model', 'call_tools']


In [52]:
# === Test the Custom Graph ===

print("Task: What is 100 * 55 + 20?")
print("=" * 50)

result = custom_graph.invoke(
    {"messages": [HumanMessage(content="What is 100 * 55 + 20?")]}
)

print("\n--- Full Message Trace ---")
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"[{role}] Requested tool calls:")
        for tc in msg.tool_calls:
            print(f"  -> {tc['name']}({tc['args']})")
    elif isinstance(msg, ToolMessage):
        print(f"[ToolMessage] {msg.content}")
    elif hasattr(msg, "content") and msg.content:
        print(f"[{role}] {msg.content[:300]}")

print("\n--- Final Answer ---")
print(result["messages"][-1].content)

Task: What is 100 * 55 + 20?

--- Full Message Trace ---
[HumanMessage] What is 100 * 55 + 20?
[AIMessage] 5520

--- Final Answer ---
5520


---

## Section 8: FastAPI Integration

Our existing FastAPI server (`api/agentic_server.py`) exposes a ReAct agent at `POST /v1/agent/run`. We can add a **new endpoint** that uses a LangChain-based agent instead of our custom implementation.

### Why Integrate LangChain into FastAPI?

- **Standardization:** LangChain provides a consistent interface, making the code easier to maintain
- **Ecosystem:** Access to hundreds of prebuilt tools, document loaders, retrievers, etc.
- **Less custom code:** The ReAct loop, tool parsing, and error handling are all built in

Below is the code snippet you would add to `agentic_server.py` to create a new LangChain-powered endpoint. We show it as a display snippet first, then we test the existing custom endpoint.

In [53]:
# === FastAPI Integration Code Snippet ===
# This is the code you would ADD to api/agentic_server.py
# to create a new LangChain-powered endpoint.
#
# NOTE: This cell is for DISPLAY ONLY -- it shows the code pattern.
# Do not run this cell directly; it requires the FastAPI app context.

fastapi_integration_code = '''
# ============================================================
# ADD THIS TO api/agentic_server.py
# ============================================================

# --- New Imports (add to the top of the file) ---
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

# --- Define LangChain tools ---
# You can wrap the existing ToolExecutor methods as LangChain tools.

@tool
def lc_calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    executor = ToolExecutor()
    return executor._calculator(expression)

@tool
def lc_get_current_time(timezone: str = "Asia/Riyadh") -> str:
    """Get the current date and time."""
    executor = ToolExecutor()
    return executor._get_current_time(timezone)

# --- Initialize LangChain agent ---
langchain_llm = ChatOllama(
    model=Config.MODEL_NAME,
    base_url=Config.OLLAMA_BASE_URL,
    temperature=0.0,
)

langchain_tools = [lc_calculator, lc_get_current_time]
langchain_agent = create_react_agent(model=langchain_llm, tools=langchain_tools)

# --- New Endpoint ---
@app.post("/v1/langchain/agent/run")
async def langchain_agent_run(task: AgentTask):
    """Run a task using the LangChain ReAct agent."""
    import uuid
    task_id = str(uuid.uuid4())[:8]

    try:
        result = langchain_agent.invoke(
            {"messages": [HumanMessage(content=task.task)]}
        )

        # Extract tool calls from the message history
        tool_calls_log = []
        steps_log = []
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    tool_calls_log.append({
                        "tool": tc["name"],
                        "arguments": tc["args"],
                    })
            steps_log.append({
                "type": msg.__class__.__name__,
                "content": getattr(msg, "content", "")[:500],
            })

        final_answer = result["messages"][-1].content

        return AgentResponse(
            task_id=task_id,
            status="completed",
            result=final_answer,
            steps=steps_log,
            tool_calls=tool_calls_log,
            total_iterations=len(steps_log),
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

print(fastapi_integration_code)


# ============================================================
# ADD THIS TO api/agentic_server.py
# ============================================================

# --- New Imports (add to the top of the file) ---
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

# --- Define LangChain tools ---
# You can wrap the existing ToolExecutor methods as LangChain tools.

@tool
def lc_calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    executor = ToolExecutor()
    return executor._calculator(expression)

@tool
def lc_get_current_time(timezone: str = "Asia/Riyadh") -> str:
    """Get the current date and time."""
    executor = ToolExecutor()
    return executor._get_current_time(timezone)

# --- Initialize LangChain agent ---
langchain_llm = ChatOllama(
    model=Config.MODEL_NAME,
    base_url=Config.OLLAMA_BASE_URL,
    temperatur

In [54]:
# === Test the Existing Custom Agent Endpoint ===
# This calls the /v1/agent/run endpoint on our running FastAPI server.
# Make sure agentic_server.py is running on port 8000 before executing.

API_BASE_URL = "http://localhost:8000"

# First, check if the server is running
try:
    health = httpx.get(f"{API_BASE_URL}/health", timeout=5.0)
    print(f"Server health: {health.json()}")
    print()

    # Run a task through the custom agent
    task_payload = {
        "task": "What is 42 * 18 + 99?",
        "tools": ["calculator"],
        "max_iterations": 5,
    }

    print(f"Sending task: {task_payload['task']}")
    response = httpx.post(
        f"{API_BASE_URL}/v1/agent/run",
        json=task_payload,
        timeout=60.0,
    )
    response.raise_for_status()
    result = response.json()

    print(f"\nStatus      : {result['status']}")
    print(f"Result      : {result['result']}")
    print(f"Iterations  : {result['total_iterations']}")
    print(f"Tool calls  : {len(result['tool_calls'])}")
    for tc in result["tool_calls"]:
        print(f"  - {tc['tool']}({tc['arguments']}) -> {tc['result'][:100]}")

except httpx.ConnectError:
    print(f"Cannot connect to {API_BASE_URL}")
    print("Make sure the FastAPI server is running:")
    print("  cd api && python agentic_server.py")
except Exception as e:
    print(f"Error: {e}")

Server health: {'status': 'healthy', 'ollama': 'healthy', 'model': 'gpt-oss:120b-cloud', 'timestamp': '2026-02-24T04:24:05.048939'}

Sending task: What is 42 * 18 + 99?

Status      : max_iterations_reached
Result      : None
Iterations  : 5
Tool calls  : 0


---

## Section 9: Comparison -- Custom vs LangChain Agent

Now let's run the **same task** through both approaches and compare them side-by-side:

1. **Custom ReAct Agent** -- via HTTP call to our FastAPI server (`/v1/agent/run`)
2. **LangChain ReAct Agent** -- built directly in this notebook

We will measure:
- Number of iterations / steps
- Number of tool calls
- Final output quality
- Execution time

In [55]:
# === Side-by-Side Comparison ===

COMPARISON_TASK = "What is 250 * 4 + 75?"

print(f"Task: {COMPARISON_TASK}")
print("=" * 70)

# --- Approach 1: Custom ReAct Agent via FastAPI ---
print("\n[1] CUSTOM REACT AGENT (agentic_server.py via HTTP)")
print("-" * 70)

custom_result = None
custom_time = None

try:
    start = time.time()
    response = httpx.post(
        f"{API_BASE_URL}/v1/agent/run",
        json={"task": COMPARISON_TASK, "tools": ["calculator"], "max_iterations": 5},
        timeout=60.0,
    )
    response.raise_for_status()
    custom_time = time.time() - start
    custom_result = response.json()

    print(f"  Status       : {custom_result['status']}")
    print(f"  Iterations   : {custom_result['total_iterations']}")
    print(f"  Tool calls   : {len(custom_result['tool_calls'])}")
    for tc in custom_result["tool_calls"]:
        print(f"    - {tc['tool']}({tc['arguments']}) -> {tc['result'][:80]}")
    print(f"  Final answer : {custom_result['result']}")
    print(f"  Time         : {custom_time:.2f}s")

except httpx.ConnectError:
    print("  SKIPPED: FastAPI server not running at", API_BASE_URL)
    print("  Start it with: cd api && python agentic_server.py")
except Exception as e:
    print(f"  ERROR: {e}")

# --- Approach 2: LangChain ReAct Agent ---
print("\n[2] LANGCHAIN REACT AGENT (LangGraph in this notebook)")
print("-" * 70)

start = time.time()
lc_result = react_agent.invoke(
    {"messages": [HumanMessage(content=COMPARISON_TASK)]}
)
lc_time = time.time() - start

# Count tool calls and steps
lc_tool_calls = []
lc_steps = 0
for msg in lc_result["messages"]:
    lc_steps += 1
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            lc_tool_calls.append(tc)

lc_final_answer = lc_result["messages"][-1].content

print(f"  Status       : completed")
print(f"  Steps        : {lc_steps}")
print(f"  Tool calls   : {len(lc_tool_calls)}")
for tc in lc_tool_calls:
    print(f"    - {tc['name']}({tc['args']})")
print(f"  Final answer : {lc_final_answer[:200]}")
print(f"  Time         : {lc_time:.2f}s")

Task: What is 250 * 4 + 75?

[1] CUSTOM REACT AGENT (agentic_server.py via HTTP)
----------------------------------------------------------------------
  Status       : max_iterations_reached
  Iterations   : 5
  Tool calls   : 0
  Final answer : None
  Time         : 4.53s

[2] LANGCHAIN REACT AGENT (LangGraph in this notebook)
----------------------------------------------------------------------
  Status       : completed
  Steps        : 4
  Tool calls   : 1
    - calculator({'expression': '250 * 4 + 75'})
  Final answer : The result is **1075**.
  Time         : 3.97s


In [56]:
# === Print Comparison Summary ===

print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Metric':<25} {'Custom Agent':<22} {'LangChain Agent':<22}")
print("-" * 70)

if custom_result:
    print(f"{'Iterations/Steps':<25} {custom_result['total_iterations']:<22} {lc_steps:<22}")
    print(f"{'Tool Calls':<25} {len(custom_result['tool_calls']):<22} {len(lc_tool_calls):<22}")
    print(f"{'Execution Time':<25} {custom_time:.2f}s{'':<17} {lc_time:.2f}s")
    custom_answer_short = (custom_result['result'] or 'N/A')[:40]
    lc_answer_short = lc_final_answer[:40]
    print(f"{'Final Answer':<25} {custom_answer_short:<22} {lc_answer_short:<22}")
else:
    print(f"{'Iterations/Steps':<25} {'N/A (server down)':<22} {lc_steps:<22}")
    print(f"{'Tool Calls':<25} {'N/A':<22} {len(lc_tool_calls):<22}")
    print(f"{'Execution Time':<25} {'N/A':<22} {lc_time:.2f}s")
    print(f"{'Final Answer':<25} {'N/A':<22} {lc_final_answer[:40]:<22}")

print("-" * 70)


COMPARISON SUMMARY
Metric                    Custom Agent           LangChain Agent       
----------------------------------------------------------------------
Iterations/Steps          5                      4                     
Tool Calls                0                      1                     
Execution Time            4.53s                  3.97s
Final Answer              N/A                    The result is **1075**.
----------------------------------------------------------------------


---

## Summary: Custom Agent vs LangChain Agent

| Aspect | Custom Agent (agentic_server.py) | LangChain Agent (this notebook) |
|---|---|---|
| **Setup Complexity** | High -- must build OllamaClient, ToolExecutor, ReActAgent, prompt parsing | Low -- `create_react_agent()` handles everything |
| **Tool Definition** | Manual JSON schemas + ToolDefinition class | Simple `@tool` decorator with docstrings |
| **Tool Calling** | Text parsing with regex (`ACTION: tool_name(...)`) | Native structured tool calling (JSON) |
| **Memory/State** | Not built-in (stateless) | Built-in with `MemorySaver` and thread IDs |
| **Error Handling** | Must implement manually | Handled by LangGraph framework |
| **Customizability** | Full control over every step | High with custom `StateGraph`, moderate with prebuilt |
| **Dependencies** | Only `httpx`, `fastapi`, `pydantic` | `langchain`, `langgraph`, `langchain-ollama` |
| **Learning Curve** | Must understand ReAct pattern deeply | Higher-level abstractions, quicker to start |
| **Production Readiness** | Good for simple use cases | Better ecosystem for complex applications |
| **Observability** | Custom logging | LangSmith integration, built-in tracing |

### When to Use Each Approach

**Use the Custom Agent when:**
- You want minimal dependencies
- You need complete control over the prompt format and parsing
- Your use case is simple and well-defined
- You want to deeply understand how agents work internally

**Use LangChain/LangGraph when:**
- You need conversation memory, streaming, or complex tool chains
- You want to swap LLM providers easily (Ollama, OpenAI, Anthropic, etc.)
- You need RAG, document loading, or vector store integration
- You want to iterate quickly with less boilerplate
- You need production features like tracing and monitoring (LangSmith)

---

### Next Steps

- **Module 2:** Retrieval-Augmented Generation (RAG) -- connect your agent to documents
- **Module 3:** Fine-Tuning -- customize the base model for your domain
- **Module 4:** RLHF/DPO -- align the model with human preferences

**Congratulations on completing Module 1!**